# FlashEats — Class 5 Investigation
**Mission:** assemble trustworthy evidence about late deliveries.

Rules: no ML for first 90 minutes; preserve raw API responses; state your definition of late; never silently drop failures.

In [1]:
import os, zipfile, json, sqlite3, time, subprocess, sys
from pathlib import Path
import pandas as pd, requests

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = Path('/content/flasheats_classroom')
if not BASE.exists() or not (BASE / 'database' / 'flasheats.db').exists():
    from google.colab import files
    print('Upload FlashEats_Classroom_Pack.zip')
    uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith('.zip'))
    with zipfile.ZipFile(zip_name) as z: z.extractall('/content')
    extracted=Path('/content/FlashEats_Classroom_Pack')
    if extracted.exists(): extracted.rename(BASE)
print(BASE, BASE.exists())

/Users/suja/Suja's Folder/FDE/flasheats-classroom-pack True


## Challenge 1 — How large is the late-delivery problem?
Write down your definition of late, denominator, cancellation policy, and missing-timestamp policy before querying.

**Definition of late:** `actual_delivery_at` > `promised_eta`

**Denominator:** Total number of delivered orders (ignoring cancellations).

**Cancellation policy:** Ignore cancelled orders.

**Missing-timestamp policy:** Exclude orders missing `actual_delivery_at` or `promised_eta`.

In [2]:
con=sqlite3.connect(BASE/'database'/'flasheats.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)

,name
0,customers
1,drivers
2,restaurants
3,orders


In [3]:
query='''
SELECT COUNT(*) AS total_delivered,
       SUM(CASE WHEN datetime(actual_delivery_at) > datetime(promised_eta) THEN 1 ELSE 0 END) AS late_orders,
       SUM(CASE WHEN datetime(actual_delivery_at) > datetime(promised_eta) THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS late_rate
FROM orders
WHERE final_status = 'delivered'
  AND actual_delivery_at IS NOT NULL
  AND promised_eta IS NOT NULL;
'''
pd.read_sql(query,con)


,total_delivered,late_orders,late_rate
0,1493,841,0.563295


### Checkpoint
Compare counts with another team. If they differ, investigate row grain, duplicates, cancellations, missing timestamps, and definitions.

## Challenge 2 — Operations says traffic is the cause
Test the claim. Treat traffic/weather/distance as descriptive signals, not causal proof.

In [4]:
query_traffic='''
SELECT traffic_bucket, weather_bucket, COUNT(*) AS total,
       SUM(CASE WHEN datetime(actual_delivery_at) > datetime(promised_eta) THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS late_rate,
       AVG(distance_km_estimate) as avg_dist_km
FROM orders
WHERE final_status = 'delivered'
GROUP BY traffic_bucket, weather_bucket
ORDER BY traffic_bucket;
'''
pd.read_sql(query_traffic, con)


,traffic_bucket,weather_bucket,total,late_rate,avg_dist_km
0,HIGH,clear,3,0.666667,10.663333
1,high,clear,337,0.626113,14.357982
2,high,heavy_rain,20,0.800000,14.577000
3,high,rain,75,0.706667,15.209200
4,low,clear,297,0.468013,14.443872
5,low,heavy_rain,12,0.666667,15.867500
6,low,rain,57,0.508772,15.395088
7,medium,clear,484,0.464876,14.467417
8,medium,heavy_rain,36,0.694444,14.906389
9,medium,rain,82,0.634146,14.696829


## Challenge 3 — Customer Support disagrees
Inspect support tickets. What are customers actually complaining about? Are ticket records unique?

In [5]:
tickets = pd.read_csv(BASE/'data'/'support_tickets.csv')
print('Total tickets:', len(tickets))
print('Unique tickets:', len(tickets['ticket_id'].unique()))

# Clean category names
tickets['category_cleaned'] = tickets['category'].str.strip().str.lower().str.replace(' ', '_')
print('\nComplaints Breakdown:')
print(tickets['category_cleaned'].value_counts())


Total tickets: 202
Unique tickets: 201

Complaints Breakdown:
category_cleaned
late_delivery        40
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
eta_issue             1
Name: count, dtype: int64


## Challenge 4 — Retrieve Dispatch data reliably

In [6]:
!pip -q install flask
api_proc=subprocess.Popen([sys.executable,str(BASE/'api'/'mock_dispatch_api.py')],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(2)
requests.get('http://127.0.0.1:8000/health').json()

fish: Unknown command: pip
fish: 
pip -q install flask
^~^


{'service': 'flasheats-dispatch-api', 'status': 'ok'}

In [7]:
url='http://127.0.0.1:8000/dispatch/orders'
r=requests.get(url,params={'page':1,'page_size':50},timeout=10)
print(r.status_code); payload=r.json(); print(payload.keys(), len(payload.get('data',[])), payload.get('has_more'))

200
dict_keys(['data', 'has_more', 'page', 'page_size', 'total_records']) 50 True


### Your task
Implement pagination + retry + raw-page preservation. Refuse to claim success if ingestion is incomplete.

In [8]:
RAW_DIR = BASE/'student_output'/'raw_dispatch'; RAW_DIR.mkdir(parents=True, exist_ok=True)
def fetch_all_dispatch_orders():
    records = []
    page = 1
    while True:
        try:
            r = requests.get('http://127.0.0.1:8000/dispatch/orders', params={'page': page, 'page_size': 50}, timeout=10)
            if r.status_code == 200:
                payload = r.json()
                data = payload.get('data', [])
                records.extend(data)
                
                with open(RAW_DIR / f'page_{page}.json', 'w') as f:
                    json.dump(payload, f)
                
                if not payload.get('has_more'):
                    break
                page += 1
            elif r.status_code == 429:
                time.sleep(1)
            else:
                r.raise_for_status()
        except requests.RequestException as e:
            print(f'Error fetching page {page}: {e}')
            time.sleep(1)
    return records

records = fetch_all_dispatch_orders()
print(f'Fetched {len(records)} records.')


Error fetching page 3: 500 Server Error: INTERNAL SERVER ERROR for url: http://127.0.0.1:8000/dispatch/orders?page=3&page_size=50
Fetched 1600 records.


## Challenge 5 — Driver events
What is observed vs inferred? Can you find a reliable driver-arrival-at-restaurant event?

In [9]:
with open(BASE/'data'/'driver_events.json') as f: driver_events=json.load(f)
# Explore a single driver's events to find a reliable arrival event
print(json.dumps(driver_events[0]['events'][:5], indent=2))

# Observation: The 'gps_ping' events represent observed locations.
# 'assigned', 'picked_up', 'delivered' are inferred or manual status changes.
# To find driver arrival, we could look at the first gps_ping near the restaurant's coordinates,
# but since we only have these events, 'picked_up' is the closest explicit event,
# although it may be manually triggered and thus delayed.


[
  {
    "order_id": "O00162",
    "type": "assigned",
    "timestamp": "2026-08-26T17:53:25.977803"
  },
  {
    "order_id": "O00162",
    "type": "gps_ping",
    "timestamp": "2026-08-26T18:10:59.372420",
    "lat": 12.826339,
    "lon": 77.448282
  },
  {
    "order_id": "O00162",
    "type": "gps_ping",
    "timestamp": "2026-08-26T18:28:32.767037",
    "lat": 12.876013,
    "lon": 77.477707
  },
  {
    "order_id": "O00162",
    "type": "gps_ping",
    "timestamp": "2026-08-26T18:46:06.161653",
    "lat": 12.925686,
    "lon": 77.507133
  },
  {
    "order_id": "O00162",
    "type": "gps_ping",
    "timestamp": "2026-08-26T19:03:39.556270",
    "lat": 12.97536,
    "lon": 77.536558
  }
]


## Final FDE recommendation
Prepare one slide: problem size, evidence, uncertainty, missing instrumentation, next step.

**Would you build the AI delay predictor now? Why or why not?**

### Final Recommendation
No, we should not build the AI delay predictor right now. The problem size is indeed large (over 50% late rate), but the evidence is currently noisy. 
- The ETA calculations might be flawed or not accounting for prep time vs travel time.
- Operational metrics like traffic explain some delays but not all.
- Support tickets suggest many issues stem from 'eta_changed' and 'restaurant_delay'.
- We lack reliable instrumentation for when the driver actually arrives at the restaurant vs when the food is handed over.
Next step: Instrument explicit timestamps for 'driver_arrived_at_restaurant' and 'food_ready' before building any ML model.